# Capítulo 10 — Paradigmas complexos de RAG (Complex RAG Paradigms)

**Livro:** começa na página **460** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## Agentic RAG or agent-driven RAG systems · PDF p. 465

#### 🧪 Exercício: Agentic RAG: o modelo decide se busca

Ofereça ao modelo uma ferramenta `buscar_documentos(consulta)` via *function calling*. Num laço, se ele pedir a ferramenta, execute a busca (BM25) e devolva o resultado; pare quando ele responder em texto. Teste com uma pergunta que exige duas buscas ("quanto custa imprimir 5 páginas e em que dias posso doar livros?").

In [ ]:
from rank_bm25 import BM25Okapi
bm = BM25Okapi([re.findall(r"\w+", (d["titulo"] + " " + d["texto"]).lower()) for d in DOCS])
def buscar_documentos(consulta):
    top = np.argsort(-bm.get_scores(re.findall(r"\w+", consulta.lower())))[:2]
    return "\n".join(f"[{DOCS[i]['id']}] {DOCS[i]['texto']}" for i in top)
ferramentas = [{"type": "function", "function": {"name": "buscar_documentos", "description": "Busca nas regras da Biblioteca Aurora",
    "parameters": {"type": "object", "properties": {"consulta": {"type": "string"}}, "required": ["consulta"]}}}]
msgs = [{"role": "system", "content": "Use a ferramenta para buscar antes de responder. Responda em português e cite os ids."},
        {"role": "user", "content": "Quanto custa imprimir 5 páginas em preto e branco e em que dias posso doar livros?"}]
for passo in range(5):
    r = conversar(msgs, tools=ferramentas).choices[0].message
    if not r.tool_calls:
        print(r.content); break
    msgs.append(r)
    for chamada in r.tool_calls:
        arg = json.loads(chamada.function.arguments); print("🔎", arg)
        msgs.append({"role": "tool", "tool_call_id": chamada.id, "content": buscar_documentos(**arg)})
print(gasto())